In [3]:
"""
═══════════════════════════════════════════════════════════════════
RACE ANALYSIS — 2021 HMDA NY mortgage data (adapted from 2019/2020 version)
═══════════════════════════════════════════════════════════════════
Goal: regenerate race_detail.parquet for 2021 with 5-group resolution
      (White, Black, Hispanic, Asian, Other) so we can recover the
      "loan size amplification" finding for the Medium post.

Paste each block into its own Colab cell, run in order.

The "1, 2, 3, 7, 8 → filter 8" pattern matches what Final_EDA already does.
Action codes 4, 5, 6 are correctly excluded (not credit decisions).
═══════════════════════════════════════════════════════════════════
"""

# ════════════════════════════════════════════════════════════════
# CELL 1 — Reload raw 2021 data from CFPB API
# ════════════════════════════════════════════════════════════════
# ════════════════════════════════════════════════════════════════
# CELL 1 (FIXED) — Reload raw 2021, keep original positional index
# ════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

URL = "https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states=NY&years=2021&actions_taken=1,2,3,7,8"

print("Loading HMDA 2021 NY from CFPB API...")
df_raw_2021 = pd.read_csv(URL, low_memory=False)
print(f"Raw data loaded: {df_raw_2021.shape[0]:,} rows × {df_raw_2021.shape[1]} cols")

# DO NOT drop code 8 here. The cleaned data's index values are positions
# in the ORIGINAL 600K-row raw frame. The cleaned data has already
# filtered out code 8, so its indices won't point to code-8 rows.

# Show what we have (no filtering)
action_names = {1: 'Loan originated', 2: 'Approved not accepted',
                3: 'Application denied', 7: 'Preapproval denied',
                8: 'Preapproval approved not accepted'}
print("\nRaw action_taken distribution (kept all for index integrity):")
for code, count in df_raw_2021['action_taken'].value_counts().sort_index().items():
    print(f"  {code}: {count:>9,} ({count/len(df_raw_2021)*100:5.2f}%)  "
          f"{action_names.get(code, '?')}")


# ════════════════════════════════════════════════════════════════
# CELL 2 — Load cleaned data + verify index alignment
# ════════════════════════════════════════════════════════════════
df_2021_cleaned = pd.read_parquet('/content/df_2021_cleaned.parquet')

print(f"Cleaned data rows: {len(df_2021_cleaned):,}")
print(f"Raw data rows    : {len(df_raw_2021):,}")
print(f"Cleaned index range: {df_2021_cleaned.index.min()} to {df_2021_cleaned.index.max()}")

# CRITICAL: spot-check that positional indexing aligns
# Compare loan_amount on 5 sampled cleaned indices to raw at the same positions
sample_idx = df_2021_cleaned.index[:5].tolist()
print("\n=== ALIGNMENT SPOT CHECK ===")
print(f"{'Cleaned idx':>12} {'Cleaned loan':>14} {'Raw loan':>14} {'Match':>8}")
mismatches = 0
for idx in sample_idx:
    cleaned_loan = df_2021_cleaned.loc[idx, 'loan_amount']
    raw_loan = df_raw_2021.iloc[idx]['loan_amount']
    match = '✅' if abs(cleaned_loan - raw_loan) < 1 else '❌'
    if match == '❌':
        mismatches += 1
    print(f"{idx:>12} {cleaned_loan:>14,.0f} {raw_loan:>14,.0f} {match:>8}")

if mismatches > 0:
    print(f"\n⚠️  {mismatches}/{len(sample_idx)} mismatches detected.")
    print("STOP — alignment is broken. Investigate before continuing.")
else:
    print("\n✅ Alignment verified.")


# ════════════════════════════════════════════════════════════════
# CELL 3 — Build race_detail with 5-group resolution
# ════════════════════════════════════════════════════════════════
def combined_race_ethnicity(row):
    """
    HMDA convention: Hispanic ethnicity overrides race.
    Groups: White, Black, Hispanic, Asian, Other, Unknown.
    """
    eth  = str(row['derived_ethnicity']).strip()
    race = str(row['derived_race']).strip()

    is_hispanic = ('Hispanic' in eth or 'Latino' in eth) and 'Not Hispanic' not in eth
    if is_hispanic:
        return 'Hispanic or Latino'

    if race == 'White':
        return 'White'
    elif race == 'Black or African American':
        return 'Black or African American'
    elif race == 'Asian':
        return 'Asian'
    elif race in ['Native Hawaiian or Other Pacific Islander',
                  'American Indian or Alaska Native',
                  '2 or more minority races']:
        return 'Other'
    else:
        return 'Unknown'


# Pull race/ethnicity from raw using the cleaned index
raw_subset = df_raw_2021.iloc[df_2021_cleaned.index][
    ['derived_race', 'derived_ethnicity']
].copy()
raw_subset.index = df_2021_cleaned.index
raw_subset['race_ethnicity'] = raw_subset.apply(combined_race_ethnicity, axis=1)

# Build race_detail.parquet
race_detail_df = pd.DataFrame({
    'derived_race'   : raw_subset['derived_race'].values,
    'race_binary'    : (raw_subset['derived_race'] == 'White').astype(int).values,
    'race_ethnicity' : raw_subset['race_ethnicity'].values,
}, index=df_2021_cleaned.index)

print("Race/ethnicity distribution (5 groups + Unknown):")
print(race_detail_df['race_ethnicity'].value_counts())
print(f"\nTotal rows: {len(race_detail_df):,}")
print(f"\nBinary breakdown (matches race_meta.parquet):")
print(race_detail_df['race_binary'].value_counts())

race_detail_df.to_parquet('race_detail.parquet')
print("\n✅ Saved race_detail.parquet")


# ════════════════════════════════════════════════════════════════
# CELL 4 — Restore the test-set arrays needed for subgroup analysis
# (matches the original FairLens train/test split exactly)
# ════════════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split

cost_meta = pd.read_parquet('cost_meta.parquet')
race_meta = pd.read_parquet('race_meta.parquet')

drop_cols = ['loan_approved', 'race_binary']
for col in ['race_group', 'derived_race']:
    if col in df_2021_cleaned.columns:
        drop_cols.append(col)

X = df_2021_cleaned.drop(columns=drop_cols)
y = df_2021_cleaned['loan_approved']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pull the same test indices for race + cost
race_test_idx = X_test.index
race_eth_test = race_detail_df.loc[race_test_idx, 'race_ethnicity'].values
cost_test     = cost_meta.loc[race_test_idx]
y_test_np     = y_test.values
loan_amounts   = cost_test['loan_amount'].values
interest_rates = cost_test['interest_rate'].values
loan_terms     = cost_test['loan_term_years'].values

print(f"Test set: {len(y_test_np):,} applicants")
print(f"\nSubgroup counts in test set:")
print(pd.Series(race_eth_test).value_counts())


# ════════════════════════════════════════════════════════════════
# CELL 5 — Load pre-trained model predictions from test_probs.parquet
# (These are the same probabilities the Streamlit app uses)
# ════════════════════════════════════════════════════════════════
test_probs = pd.read_parquet('test_probs.parquet')

THRESHOLD = 0.5

sub_models = []
for model_name in test_probs.columns:
    preds = (test_probs[model_name].values >= THRESHOLD).astype(int)
    sub_models.append((model_name, preds))
    print(f"Loaded {model_name}: approval rate = {preds.mean():.1%}")


# ════════════════════════════════════════════════════════════════
# CELL 6 — Subgroup dollar-cost analysis
# ════════════════════════════════════════════════════════════════
def amortized_interest_npv(loan_amount, annual_rate, term_years, discount_rate=0.03):
    r = annual_rate / 100 / 12
    d = discount_rate / 12
    n = int(term_years * 12)
    if r == 0 or n == 0:
        return 0
    mp = loan_amount * (r * (1+r)**n) / ((1+r)**n - 1)
    return sum(mp / (1+d)**t for t in range(1, n+1)) - loan_amount


def remaining_balance(loan_amount, annual_rate, term_years, default_year):
    r = annual_rate / 100 / 12
    n = int(term_years * 12)
    k = int(default_year * 12)
    if r == 0 or n == 0:
        return loan_amount
    mp = loan_amount * (r * (1+r)**n) / ((1+r)**n - 1)
    return max(loan_amount * (1+r)**k - mp * ((1+r)**k - 1) / r, 0)


GROUPS    = ['White', 'Black or African American', 'Hispanic or Latino', 'Asian']
LGD       = 0.40
DISCOUNT  = 0.03

all_results = {}

print(f"\n{'='*105}")
print(f"DOLLAR COST BY RACE/ETHNICITY SUBGROUP  (LGD={LGD}, discount={DISCOUNT})")
print(f"{'='*105}")

for model_name, preds in sub_models:
    print(f"\nModel: {model_name}")
    print(f"{'Group':<32} {'N':>7} {'FN':>7} {'FN%':>7} "
          f"{'Avg Loan':>12} {'$/app':>14} {'DI':>8} {'Gap vs White':>14}")
    print("─" * 105)

    white_per_app  = None
    white_app_rate = None
    row_data       = []

    for grp in GROUPS:
        mask = race_eth_test == grp
        if mask.sum() < 50:
            continue

        yt = y_test_np[mask]
        yp = preds[mask]
        loans_g = loan_amounts[mask]
        rates_g = interest_rates[mask]
        terms_g = loan_terms[mask]

        fn_mask = (yt == 1) & (yp == 0)
        fp_mask = (yt == 0) & (yp == 1)

        fn_cost = sum(
            amortized_interest_npv(l, r, t, DISCOUNT)
            for l, r, t in zip(loans_g[fn_mask], rates_g[fn_mask], terms_g[fn_mask])
        )
        fp_cost = sum(
            remaining_balance(l, r, t, t/2) * LGD
            for l, r, t in zip(loans_g[fp_mask], rates_g[fp_mask], terms_g[fp_mask])
        )

        per_app  = (fn_cost + fp_cost) / mask.sum()
        app_rate = yp.mean()
        fn_rate  = fn_mask.sum() / mask.sum() * 100
        avg_loan = loans_g[fn_mask].mean() if fn_mask.sum() > 0 else 0

        if grp == 'White':
            white_per_app  = per_app
            white_app_rate = app_rate

        row_data.append({
            'grp'     : grp,
            'n'       : mask.sum(),
            'fn'      : fn_mask.sum(),
            'fn_rate' : fn_rate,
            'avg_loan': avg_loan,
            'per_app' : per_app,
            'app_rate': app_rate,
        })

    all_results[model_name] = row_data

    for d in row_data:
        di     = d['app_rate'] / white_app_rate if white_app_rate and d['grp'] != 'White' else None
        gap    = d['per_app']  - white_per_app  if white_per_app   and d['grp'] != 'White' else None
        di_str = f"{di:>7.3f}"        if di  is not None else "  (ref)"
        gs     = f"${gap:>+12,.0f}"   if gap is not None else f"{'(baseline)':>14}"
        flag   = " ⚠️" if di is not None and di < 0.8 else ""
        print(f"{d['grp']:<32} {d['n']:>7,} {d['fn']:>7,} {d['fn_rate']:>6.1f}% "
              f"${d['avg_loan']:>11,.0f} ${d['per_app']:>13,.2f} {di_str}{gs}{flag}")
    print("─" * 105)


# ════════════════════════════════════════════════════════════════
# CELL 7 — The headline: loan-size amplification finding
# ════════════════════════════════════════════════════════════════
# Pick the LR Baseline model (most interpretable for the writeup)
model_for_callout = next(
    (m for m in test_probs.columns if 'Baseline' in m),
    list(test_probs.columns)[0]
)
data_for_callout = {d['grp']: d for d in all_results[model_for_callout]}

print(f"\n{'='*75}")
print(f"NOVEL FINDING — LOAN-SIZE AMPLIFICATION (model: {model_for_callout})")
print(f"{'='*75}")

if all(g in data_for_callout for g in ['White', 'Asian', 'Black or African American']):
    w = data_for_callout['White']
    a = data_for_callout['Asian']
    b = data_for_callout['Black or African American']

    print(f"\nAverage loan size among WRONGFULLY DENIED applicants:")
    print(f"  White : ${w['avg_loan']:>12,.0f}")
    print(f"  Black : ${b['avg_loan']:>12,.0f}  "
          f"({(b['avg_loan']/w['avg_loan']-1)*100:+.1f}% vs White)")
    print(f"  Asian : ${a['avg_loan']:>12,.0f}  "
          f"({(a['avg_loan']/w['avg_loan']-1)*100:+.1f}% vs White)")

    print(f"\nFN rate (wrongful denial rate):")
    print(f"  White : {w['fn_rate']:5.2f}%")
    print(f"  Black : {b['fn_rate']:5.2f}%")
    print(f"  Asian : {a['fn_rate']:5.2f}%")

    asian_gap = a['per_app'] - w['per_app']
    black_gap = b['per_app'] - w['per_app']
    print(f"\nPer-applicant dollar gap vs White:")
    print(f"  Black gap: ${black_gap:>+9,.0f}")
    print(f"  Asian gap: ${asian_gap:>+9,.0f}")

    if abs(asian_gap) > abs(black_gap):
        print(f"\n  ✅ Asian dollar gap EXCEEDS Black dollar gap")
        print(f"  Even though Asian FN rate may be lower, "
              f"the LARGER avg loan size amplifies the dollar harm.")
    else:
        print(f"\n  ⚠️  Asian gap is not larger than Black gap in this run — "
              f"the amplification effect may need re-evaluation for 2021.")

    print(f"""
KEY INSIGHT:
Rate-parity fairness metrics (DI, EOD, FPR_diff) measure approval
rates but ignore DOLLAR MAGNITUDE. A group with a lower wrongful
denial rate can still suffer the largest dollar gap if their
average loan size is bigger.

This is the loan-size amplification effect — and it's invisible
to every traditional fairness metric. It's the same insight as
the binary Fairness Paradox, applied at the subgroup level.
""")

Loading HMDA 2021 NY from CFPB API...
Raw data loaded: 600,611 rows × 99 cols

Raw action_taken distribution (kept all for index integrity):
  1:   474,930 (79.07%)  Loan originated
  2:    19,549 ( 3.25%)  Approved not accepted
  3:   105,159 (17.51%)  Application denied
  7:       242 ( 0.04%)  Preapproval denied
  8:       731 ( 0.12%)  Preapproval approved not accepted
Cleaned data rows: 477,937
Raw data rows    : 600,611
Cleaned index range: 7 to 600610

=== ALIGNMENT SPOT CHECK ===
 Cleaned idx   Cleaned loan       Raw loan    Match
           7        265,000        265,000        ✅
           9        245,000        245,000        ✅
          10        475,000        475,000        ✅
          11        175,000        175,000        ✅
          14      1,805,000      1,805,000        ✅

✅ Alignment verified.
Race/ethnicity distribution (5 groups + Unknown):
race_ethnicity
White                        350646
Asian                         54169
Black or African American     37382

In [4]:
# In Colab
from google.colab import files
files.download('race_detail.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# Save test-aligned race ethnicity for the Streamlit app
import pandas as pd
race_eth_df = pd.DataFrame({'race_ethnicity': race_eth_test})
race_eth_df.to_parquet('race_eth_test.parquet')
print(f"Saved race_eth_test.parquet: {len(race_eth_df):,} rows")
print(f"Distribution:")
print(race_eth_df['race_ethnicity'].value_counts())

# Download
from google.colab import files
files.download('race_eth_test.parquet')

Saved race_eth_test.parquet: 95,588 rows
Distribution:
race_ethnicity
White                        70150
Asian                        10874
Black or African American     7369
Hispanic or Latino            6692
Other                          503
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>